# SHIPP — Validate ORS Evidence

**Review draft — read-only.** This notebook does not call OpenRouteService and does not write tables.

It validates the evidence produced by Stage 6/7:

`silver_candidate_pairs → ORS raw responses → silver_routes`

## Important repo preflight
The current repository uses `SILVER_CANDIDATE_PAIRS` and `BRONZE_ROUTE_RESPONSES`
in development notebooks, but those constants are not currently defined in
`config/tables.py`.

This notebook uses the expected physical names so you can inspect runtime state,
but the final gate deliberately remains **BLOCKED** until those constants are
centralized in `config/tables.py`.

In [ ]:
%run ../common/shipp_silver_lib

In [ ]:
import config.tables as table_config

EXPECTED_TABLES = {
    "SILVER_CANDIDATE_PAIRS": f"{CATALOG}.{SILVER_SCHEMA}.silver_candidate_pairs",
    "BRONZE_ROUTE_RESPONSES": f"{CATALOG}.{BRONZE_SCHEMA}.ors_route_responses",
    "SILVER_ROUTES": SILVER_ROUTES,
}

missing_config_constants = [
    name for name in ("SILVER_CANDIDATE_PAIRS", "BRONZE_ROUTE_RESPONSES")
    if not hasattr(table_config, name)
]

SILVER_CANDIDATE_PAIRS_RUNTIME = getattr(
    table_config, "SILVER_CANDIDATE_PAIRS", EXPECTED_TABLES["SILVER_CANDIDATE_PAIRS"]
)
BRONZE_ROUTE_RESPONSES_RUNTIME = getattr(
    table_config, "BRONZE_ROUTE_RESPONSES", EXPECTED_TABLES["BRONZE_ROUTE_RESPONSES"]
)

print("Repo contract check:")
print("  missing config constants:", missing_config_constants or "none")
print("  candidate pairs table:   ", SILVER_CANDIDATE_PAIRS_RUNTIME)
print("  ORS Bronze table:        ", BRONZE_ROUTE_RESPONSES_RUNTIME)
print("  Silver routes table:     ", SILVER_ROUTES)

In [ ]:
required_runtime_tables = [
    SILVER_CANDIDATE_PAIRS_RUNTIME,
    BRONZE_ROUTE_RESPONSES_RUNTIME,
    SILVER_ROUTES,
]
missing_runtime_tables = [t for t in required_runtime_tables if not table_exists(t)]

for t in required_runtime_tables:
    print(" ", "OK " if table_exists(t) else "MISSING", t)

assert not missing_runtime_tables, (
    "BLOCKED: run Stage 6/7 writers first. Missing runtime table(s): "
    + ", ".join(missing_runtime_tables)
)

In [ ]:
bronze = spark.table(BRONZE_ROUTE_RESPONSES_RUNTIME)
routes = spark.table(SILVER_ROUTES)
pairs = spark.table(SILVER_CANDIDATE_PAIRS_RUNTIME)

pair_keys = pairs.select("origin_key", "dest_key", "profile").distinct()
covered_keys = routes.select("origin_key", "dest_key", "profile").distinct()
uncovered = pair_keys.join(covered_keys, ["origin_key", "dest_key", "profile"], "left_anti")

successful_raw = bronze.filter(
    (F.col("http_status") == 200) & F.col("error_message").isNull()
)
failed_raw = bronze.filter(
    (F.col("http_status") != 200) | F.col("http_status").isNull() | F.col("error_message").isNotNull()
)

invalid_route_status = routes.filter(~F.col("route_status").isin("OK", "NO_ROUTE")).count()
duplicate_route_key = duplicate_key_count(routes, "origin_key", "dest_key", "profile")
negative_distance = routes.filter(F.col("distance_km") < 0).count()
negative_duration = routes.filter(F.col("duration_min") < 0).count()
ok_without_distance = routes.filter(
    (F.col("route_status") == "OK") & F.col("distance_km").isNull()
).count()
no_route_with_distance = routes.filter(
    (F.col("route_status") == "NO_ROUTE") & F.col("distance_km").isNotNull()
).count()

orphan_source_batches = (
    routes.select(F.col("source_batch_id").alias("batch_id")).distinct()
    .join(successful_raw.select("batch_id").distinct(), "batch_id", "left_anti")
    .count()
)

print("Candidate route keys:             ", pair_keys.count())
print("Successful raw ORS responses:     ", successful_raw.count())
print("Historical failed raw responses:  ", failed_raw.count())
print("Silver route rows:                ", routes.count())
print("Uncovered candidate route keys:   ", uncovered.count())
print("Orphan source_batch_id values:    ", orphan_source_batches)

In [ ]:
display(
    bronze.select(
        "called_at", "batch_id", "dest_key",
        F.size("origin_keys").alias("origin_count"),
        "http_status", "attempts", "latency_ms", "error_message"
    )
    .orderBy(F.col("called_at").desc())
    .limit(20)
)

if uncovered.count():
    print("UNRESOLVED route keys:")
    display(uncovered.limit(100))

In [ ]:
run_quality_gate({
    "duplicate route key": duplicate_route_key,
    "invalid route_status": invalid_route_status,
    "negative distance": negative_distance,
    "negative duration": negative_duration,
    "OK route without distance": ok_without_distance,
    "NO_ROUTE with distance": no_route_with_distance,
    "route row with no successful Bronze source batch": orphan_source_batches,
    "candidate route key not covered": uncovered.count(),
}, row_count=routes.count(), allow_empty=False)

assert pair_keys.count() > 0, "EVIDENCE FAILED: zero candidate route keys."
assert successful_raw.count() > 0, "EVIDENCE FAILED: no successful ORS response is persisted in Bronze."

# Repo architecture gate: do not hide configuration drift.
assert not missing_config_constants, (
    "REPO CONTRACT BLOCKER: add these centralized constants to config/tables.py before promotion: "
    + ", ".join(missing_config_constants)
)

print("PASS — ORS evidence is complete and repo table contracts are centralized.")

In [ ]:
summary = {
    "stage": "ors_validation",
    "candidate_route_keys": pair_keys.count(),
    "successful_raw_calls": successful_raw.count(),
    "historical_failed_calls": failed_raw.count(),
    "silver_routes": routes.count(),
    "uncovered_route_keys": uncovered.count(),
    "missing_config_constants": missing_config_constants,
    "status": "PASS",
}
print(json.dumps(summary, indent=2, default=str))